# Notebook 01 — PyTerrier on tiny data (warm-up)

WIR 2026 · TH Köln — this notebook *mirrors* the official course tutorial
`pyterrier-intro` ([irgroup-classrooms/wir-2026](https://github.com/irgroup-classrooms/wir-2026)),
step for step, on the same `ai.json` collection.

**Goal.** Before we touch the 870k LongEval-Sci papers (Assignment III), learn the
PyTerrier machinery on a *tiny* collection so the real baselines in notebook 03 are
easy to read. We follow exactly the professor's path:

1. load & clean `ai.json` (keep the `Publication` records),
2. build an inverted index and read the lexicon (term `Nt` / `TF`),
3. run the `Tf` then `TF_IDF` retrievers, and
4. recompute IDF — and the full TF-IDF score — by hand from the index for learning purposes.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyterrier as pt

# PyTerrier runs on the JVM. Start it once (idempotent).
if not pt.java.started():
    pt.java.init()

# Resolve the repo root whether the kernel's cwd is the repo or the notebooks/ folder.
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_DIR = REPO_ROOT / "data"
print("repo root:", REPO_ROOT)

repo root: C:\Users\rafae\OneDrive\Desktop\TH_Koeln\03-Semester\Web_Information_Retrieval\WIR_Retriever_Engine


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## 1. Load & clean `ai.json`

Exactly as in the tutorial: we keep only the `Publication` records and then drop columns that are mostly empty.

In [3]:
with open(DATA_DIR / "ai.json", encoding="utf-8") as f:
    items = json.load(f)["items"]

df = pd.DataFrame(items)
df = df[df["type"] == "Publication"]            # keep publications
df = df.loc[:, df.isna().mean() < 0.5]          # drop columns that are >50% empty
print(f"{len(df)} records, {df.shape[1]} non-sparse columns")
df.head()

1000 records, 21 non-sparse columns


,type,id,tags,intraHash,label,user,description,date,changeDate,count,...,interHash,pub-type,journal,year,author,authors,volume,pages,abstract,bibtexKey
1000,Publication,https://www.bibsonomy.org/bibtex/24e338b04bc2a...,[710-714],4e338b04bc2abafd9fa0a1f4fe79103a,Artificial Intelligence in Agriculture,ijtsrd,,2021-04-13 13:15:04,2021-04-13 13:15:04,1,...,4d2f7caf67c19f0ff98f5cb8a04fb14d,article,International Journal of Trend in Scientific R...,2021,[Matthew N. O. Sadiku | Sarhan M. Musa | Abayo...,[{'first': 'Matthew N. O. Sadiku | Sarhan M. M...,5,710-714,Artificial Intelligence is one of the emerging...,conf/icegov/MalhotraA20
1001,Publication,https://www.bibsonomy.org/bibtex/20d06c74438c2...,[structured],0d06c74438c252c894641a8dc90260da,Kernels for structured data,ahmedjawwad4u,Amazon.com: Kernels For Structured Data (Serie...,2012-02-27 14:08:11,2012-02-27 14:08:11,1,...,7e26d26378a5534abb1073e2224102ce,book,NaN,2008,"[Thomas Gartner, World Scientific.]","[{'first': 'Thomas', 'last': 'Gartner'}, {'fir...",NaN,NaN,NaN,gartner2008kernels
1002,Publication,https://www.bibsonomy.org/bibtex/283c4174f111d...,"[digitalisation, artificial_intelligence, soci...",83c4174f111d968e8ca51b252f8fea3e,"Artificial intelligence, labour and society",meneteqel,,2024-04-08 09:45:09,2024-04-08 09:46:54,1,...,8ad322616f308bddd792b86c2d5647ac,book,NaN,2024,NaN,NaN,NaN,NaN,The rapid expansion of artificial intelligence...,poncedelcastillo2024artificial
1003,Publication,https://www.bibsonomy.org/bibtex/2cbb6efd8274a...,[dblp],cbb6efd8274af7d08c52526384fd12ce,Artifical intelligence overview.,dblp,,2019-08-20 00:00:00,2020-07-24 01:02:41,1,...,4e134fcc2174bc3ed230de57aec97642,inproceedings,NaN,1978,[Robert Balzer],"[{'first': 'Robert', 'last': 'Balzer'}]",47,225-226,NaN,conf/afips/Balzer78
1004,Publication,https://www.bibsonomy.org/bibtex/2e4bfec930752...,[dblp],e4bfec930752d7f5a445ec2f60ae0be5,Artifical Intelligence: A One-Hour Course.,dblp,dblp,2003-03-25,2003-03-25 00:00:00,1,...,733c60c08892ce18fc0ded70138c77fe,inproceedings,NaN,1986,[Robert Trappl],"[{'first': 'Robert', 'last': 'Trappl'}]",NaN,5-30,NaN,conf/iiasa/Trappl86a


## 2. Give PyTerrier the two columns it requires: `docno` and `text`

Every PyTerrier document needs a unique string id (`docno`) and the searchable
`text`. Here the title (`label`) is the text we index.

In [4]:
df["docno"] = df["id"].astype(str)
df["text"] = df["label"].fillna("")
df[["docno", "text"]].head()

,docno,text
1000,https://www.bibsonomy.org/bibtex/24e338b04bc2a...,Artificial Intelligence in Agriculture
1001,https://www.bibsonomy.org/bibtex/20d06c74438c2...,Kernels for structured data
1002,https://www.bibsonomy.org/bibtex/283c4174f111d...,"Artificial intelligence, labour and society"
1003,https://www.bibsonomy.org/bibtex/2cbb6efd8274a...,Artifical intelligence overview.
1004,https://www.bibsonomy.org/bibtex/2e4bfec930752...,Artifical Intelligence: A One-Hour Course.


## 3. Let's Index it

`IterDictIndexer` streams a list/iterator of dicts into a Terrier index. We pass an
absolute path because Terrier otherwise resolves relative paths against its own
internal `./var` home (a classic first-time gotcha).

In [5]:
index_path = REPO_ROOT / "ai_index"
index_path.mkdir(exist_ok=True)
for p in index_path.glob("*"):       # clear any stale index files from a previous run
    try:
        p.unlink()
    except OSError:
        pass

indexer = pt.IterDictIndexer(str(index_path), overwrite=True,
                             meta={"docno": 256, "text": 4096})
index_ref = indexer.index(df[["docno", "text"]].to_dict(orient="records"))
index = pt.IndexFactory.of(index_ref)
print(index.getCollectionStatistics().toString())

Number of documents: 1000
Number of terms: 1697
Number of postings: 6890
Number of fields: 0
Number of tokens: 7054
Field names: []
Positions:   false



## 4. Inspect the inverted index (the lexicon / TDM)

For each term the lexicon stores Nt = in how many documents the term occurs
(document frequency) and TF = how often it occurs in total. Let's look at the
most frequent terms — note how the top of the list is dominated by common words
(this is exactly why we need IDF and, later, stopword removal).

In [15]:
term_freq = {kv.getKey(): kv.getValue().getFrequency() for kv in index.getLexicon()}
top = sorted(term_freq.items(), key=lambda x: x[1], reverse=True)[:25]
pd.DataFrame(top, columns=["term", "total_frequency"])

,term,total_frequency
0,intellig,711
1,artifici,627
2,ai,108
3,base,82
4,learn,80
5,system,80
6,us,71
7,applic,69
8,approach,56
9,machin,45


## 5. The first retriever — `Tf`

The raw term-frequency model scores a document by how often the query terms appear
in it, with no normalisation:

$$score(d, q) = \sum_{t \in q} tf_{t,d}$$

It is the weakest model — long documents and common words win unfairly.

In [16]:
tf = pt.terrier.Retriever(index, wmodel="Tf")
tf.search("Phishing detection nlp").head()  # search returns a DataFrame with columns `qid`, `docid`, `docno`,`rank`, `score` and `query`

,qid,docid,docno,rank,score,query
0,1,860,https://www.bibsonomy.org/bibtex/2dd06a15143b3...,0,2.0,Phishing detection nlp
1,1,84,https://www.bibsonomy.org/bibtex/28580d3236777...,1,1.0,Phishing detection nlp
2,1,114,https://www.bibsonomy.org/bibtex/2efe3507995fb...,2,1.0,Phishing detection nlp
3,1,117,https://www.bibsonomy.org/bibtex/2376a899e3276...,3,1.0,Phishing detection nlp
4,1,143,https://www.bibsonomy.org/bibtex/2730b2a2b88d0...,4,1.0,Phishing detection nlp


## 6. Let's add IDF — the `TF_IDF` model

TF-IDF down-weights terms that appear in many documents (low information) and
rewards rare, discriminating terms:

$$score(d, q) = \sum_{t \in q} tf_{t,d}\cdot \log\frac{N}{df_t}$$

The ranking changes vs. plain `Tf` — documents that match the rarer query term
get promoted.

In [28]:
tfidf = pt.terrier.Retriever(index, wmodel="TF_IDF")
tfidf.search("Phishing detection nlp").head(10)

,qid,docid,docno,rank,score,query
0,1,860,https://www.bibsonomy.org/bibtex/2dd06a15143b3...,0,7.701366,Phishing detection nlp
1,1,594,https://www.bibsonomy.org/bibtex/23ff5930675d0...,1,3.265597,Phishing detection nlp
2,1,455,https://www.bibsonomy.org/bibtex/223c436aa3f23...,2,3.050601,Phishing detection nlp
3,1,597,https://www.bibsonomy.org/bibtex/21b8a21d28438...,3,3.050601,Phishing detection nlp
4,1,743,https://www.bibsonomy.org/bibtex/276fee135c8c1...,4,3.050601,Phishing detection nlp
5,1,387,https://www.bibsonomy.org/bibtex/28cff9d35fb12...,5,2.862166,Phishing detection nlp
6,1,519,https://www.bibsonomy.org/bibtex/28cff9d35fb12...,6,2.862166,Phishing detection nlp
7,1,520,https://www.bibsonomy.org/bibtex/28cff9d35fb12...,7,2.862166,Phishing detection nlp
8,1,684,https://www.bibsonomy.org/bibtex/2b121f73f40b6...,8,2.862166,Phishing detection nlp
9,1,887,https://www.bibsonomy.org/bibtex/29f1faaefa2f7...,9,2.862166,Phishing detection nlp


In [27]:
tfidf.search("system").head(10)

,qid,docid,docno,rank,score,query
0,1,524,https://www.bibsonomy.org/bibtex/24768ef52a6c0...,0,2.725956,system
1,1,513,https://www.bibsonomy.org/bibtex/24405b8995836...,1,2.689649,system
2,1,522,https://www.bibsonomy.org/bibtex/2db4a780c41c5...,2,2.689649,system
3,1,276,https://www.bibsonomy.org/bibtex/28b6f3d4fdad8...,3,2.500092,system
4,1,291,https://www.bibsonomy.org/bibtex/232a9dab69513...,4,2.500092,system
5,1,354,https://www.bibsonomy.org/bibtex/2e83a74c842ae...,5,2.500092,system
6,1,462,https://www.bibsonomy.org/bibtex/2e3f41f4affdf...,6,2.500092,system
7,1,463,https://www.bibsonomy.org/bibtex/2e3f41f4affdf...,7,2.500092,system
8,1,562,https://www.bibsonomy.org/bibtex/2a090b89fea8b...,8,2.500092,system
9,1,185,https://www.bibsonomy.org/bibtex/2608997777026...,9,2.335494,system


## 7. TF-IDF by hand

we try to recompute the scores ourselves straight from the index for learning purposes.
First the IDF of a single term, then the full TF-IDF score of one document against a query,
reading term frequencies from the inverted index. We stem terms first because the
index is stemmed.

In [41]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def idf_of(term: str) -> float:
    """let's compute the IDF of a single term"""
    lex = index.getLexicon()
    stem = stemmer.stem(term.lower())
    if stem not in lex:
        return float("nan")
    df_t = lex[stem].getDocumentFrequency()
    N = index.getCollectionStatistics().getNumberOfDocuments()
    return float(np.log10(N / df_t))

for t in ["Phishing", "detection", "nlp", "learn"]:
    print(f"  idf({t:<8}) = {idf_of(t):.3f}")

  idf(Phishing) = 3.000
  idf(detection) = 1.469
  idf(nlp     ) = nan
  idf(learn   ) = 1.125


In [48]:
def calc_tf_idf(query, docno: str, index, stemmer=stemmer) -> float:
    """TF-IDF score of a single document (docno) against a query, computed by hand."""
    terms = query.lower().split() if isinstance(query, str) else [t.lower() for t in query]
    lex  = index.getLexicon()
    inv  = index.getInvertedIndex()
    meta = index.getMetaIndex()
    N    = index.getCollectionStatistics().getNumberOfDocuments()
    internal_docid = meta.getDocument("docno", docno)

    score = 0.0
    for term in terms:
        stem = stemmer.stem(term)
        if stem not in lex:
            continue
        entry = lex[stem]
        df_t  = entry.getDocumentFrequency()

        # walk this term's postings to find our document's raw term frequency
        tf = 0
        for posting in inv.getPostings(entry):
            if posting.getId() == internal_docid:
                tf = posting.getFrequency()
                break

        if tf:
            score += tf * float(np.log10(N / df_t))   # tf * idf, log base 10
    return score

# Let's test it :)
docno = df["docno"].iloc[0]
query = "Artificial intelligence"
print(f"TF-IDF(doc={docno[:48]}..., q='{query}') = {calc_tf_idf(query, docno, index):.4f}")

TF-IDF(doc=https://www.bibsonomy.org/bibtex/24e338b04bc2aba..., q='Artificial intelligence') = 0.3943


> **Takeaways:** We can index now, read the lexicon, and we understand TF vs.
> TF-IDF (IDF rewards rare terms; document length is still unhandled — that is what
> BM25 fixes in notebook 03). Next: `02_inspect_and_index` does the same on the
> real LongEval-Sci collection.